# Step 7 — Diagnostics, error analysis and insights

Where the model helps and why: segment analysis, behavioural insights, SHAP, tower attention and feature-group ablations.

### Setup (run first)

**On Google Colab:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Put the two raw JDsearch
files in `MyDrive/bt4222/raw/` (`user_behavior_data.txt`, `product_meta_data.txt`). The cell mounts Drive, clones the
repository, installs the requirements and writes every output to `MyDrive/bt4222/outputs/`, so results persist between
sessions and the notebooks can be run one after another (or on different days).

**Locally:** run the notebook from the repository with the virtual environment active; data is read from `data/raw/`
and outputs go to `outputs/`.

`CONFIG = "configs/smoke.yaml"` runs everything on a 2,000-user sample with tiny models (minutes, CPU) to check the code;
`configs/default.yaml` is the real run. Every stage skips work whose outputs already exist; pass `"--force"` to redo one.

In [ ]:
import os, subprocess, sys
from pathlib import Path

CONFIG = "configs/default.yaml"          # "configs/smoke.yaml" for a quick check
REPO_URL = "https://github.com/clarequek/bt4222-two-tower-recsys.git"
DRIVE_DIR = "/content/drive/MyDrive/bt4222"   # Colab: raw/ (input) and outputs/ (written by the pipeline)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/bt4222-two-tower-recsys")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from jds.notebook import Runner, banner
run = Runner(CONFIG, drive_dir=DRIVE_DIR if IN_COLAB else None)

## 7.1 Run (`scripts/07_diagnostics.py`, `src/jds/diagnostics.py`)

* **segments** — test metrics by user activity in the click window, history length, query frequency and length, whether
  the positives are cold items, and the deepest label; ranker vs two-tower vs popularity vs the prior team's model.
* **insights** — lift of behavioural signals (open carts, repurchase, recent clicks, brand/shop loyalty, similar past
  queries, cold items) on engagement in the test session.
* **shap** — exact TreeSHAP contributions of the ranker, by feature and feature group.
* **attention** — how much query-guided attention the tower puts on each history segment.
* **ablations** (optional, ~1 h) — rankers retrained without one feature group at a time.

In [ ]:
run.stage("07_diagnostics.py", "--parts", "attention,segments,insights,shap")

In [ ]:
banner()
run.figure("diag_segments.png")
run.table("diag_segments.csv", highlight=("mrr", "gndcg10"))

In [ ]:
run.table("diag_insights.csv", highlight=("pos_lift",), caption="Behavioural signals vs engagement in the test session")

In [ ]:
run.figure("diag_shap.png")
run.table("diag_shap_groups.csv", highlight=())

In [ ]:
run.json("diag_attention.json")

In [ ]:
RUN_ABLATIONS = False
if RUN_ABLATIONS:
    run.stage("07_diagnostics.py", "--parts", "ablations")
run.table("diag_ablations.csv", caption="Ranker without one feature group (30% of training users)")